# ਪਾਠ 08 - ਮਲਟੀ-ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ


## ਸੈਟਅਪ


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਬਹੁ-ਏਜੰਟ ਸਿਸਟਮ ਕਿਉਂ?

ਅਸਲੀ ਦੁਨੀਆਂ ਦੇ ਕਾਰਜ ਜਿਵੇਂ ਕਿ ਯਾਤਰਾ ਦੀ ਯੋਜਨਾ ਬਣਾਉਣਾ ਕਈ ਵੱਖ-ਵੱਖ ਤਰ੍ਹਾਂ ਦੀ ਮਹਿਰਤ ਮੰਗਦਾ ਹੈ — ਲੋਜਿਸਟਿਕਸ, ਸਥਾਨਕ ਜਾਣਕਾਰੀ, ਬਜਟ ਬਨਾਉਣਾ, ਅਤੇ ਹੋਰ। ਇਕੱਲਾ ਏਜੰਟ ਜੋ ਸਾਰਾ ਕੰਮ ਸੰਭਾਲਣ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ, ਜਲਦੀ ਹੀ ਅਸਮਰੱਥ ਹੋ ਜਾਂਦਾ ਹੈ।

ਬਹੁ-ਏਜੰਟ ਸਿਸਟਮ ਇਸ ਦੇ ਹੱਲ ਵਜੋਂ **ਤਖਸਾਸੀ** ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹਨ: ਹਰ ਏਜੰਟ ਇੱਕ ਖੇਤਰ ਤੇ ਧਿਆਨ ਦਿੰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਇੱਕ ਆਮ ਏਜੰਟ ਦੀ ਤુલਨਾ ਵਿੱਚ ਬਿਹਤਰ ਗੁਣਵੱਤਾ ਵਾਲੇ ਨਤੀਜੇ ਨਿਕਲਦੇ ਹਨ। ਇਹ ਸਿਸਟਮ **ਸਕੈਲਬਿਲਿਟੀ** ਨੂੰ ਵੀ ਸੁਧਾਰਦਾ ਹੈ — ਤੁਸੀਂ ਨਵੇਂ ਏਜੰਟ (ਜਿਵੇਂ, ਇੱਕ ਉਡਾਣ ਵਿਸ਼ੇਸ਼ਜ ਞ, ਇੱਕ ਰੈਸਟੋਰੈਂਟ ਆਲੋਚਕ) ਸ਼ਾਮਲ ਕਰ ਸਕਦੇ ਹੋ ਬਿਨਾਂ ਮੌਜੂਦਾ ਕਾਰਜ ਪ੍ਰਣਾਲੀ ਨੂੰ ਦੁਬਾਰਾ ਲਿਖੇ। ਏਜੰਟ ਇੱਕ ਸੰਰਚਿਤ ਪ੍ਰਕਿਰਿਆ ਰਾਹੀਂ ਮਿਲਕੇ ਕੰਮ ਕਰਦੇ ਹਨ, ਇਕ ਤੋਂ ਦੂਜੇ ਵੱਲ ਸੰਦਰਭ ਸੌਂਪਦੇ ਹੋਏ। 


## ਵਿਸ਼ੇਸ਼ੀਕ੍ਰਿਤ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ਇੱਕ ਕ੍ਰਮਵਾਰ ਵਰਕਫਲੋ ਬਣਾਉਣਾ

`WorkflowBuilder` ਤੁਹਾਨੂੰ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਦਿਸ਼ਾ-ਨਿਰਦੇਸ਼ਤ ਗ੍ਰਾਫ ਵਿੱਚ ਜੋੜਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ। ਇੱਥੇ ਅਸੀਂ ਇੱਕ ਸਧਾਰਣ ਦੋ-ਕਦਮੀ ਪਾਈਪਲਾਈਨ ਬਣਾਵਾਂਗੇ: **TravelPlanner** ਯਾਤਰਾ ਦੀ ਯੋਜਨਾ ਬਣਾਉਂਦਾ ਹੈ, ਫਿਰ **TravelConcierge** ਇਸ ਦੀ ਸਮੀਖਿਆ ਕਰਦਾ ਹੈ ਅਤੇ ਇਸਨੂੰ ਹੋਰ ਵਧੀਆ ਬਣਾਉਂਦਾ ਹੈ।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਵਰਕਫਲੋ ਵਿੱਚ ਹੋਰ ਏਜੰਟ ਸ਼ਾਮਲ ਕਰਨਾ

ਮਲਟੀ-ਏਜੰਟ ਪੈਟਰਨ ਦੇ ਸਭ ਤੋਂ ਵੱਡੇ ਫਾਏਦਿਆਂ ਵਿੱਚੋਂ ਇੱਕ ਇਹ ਹੈ ਕਿ ਇਸ ਨੂੰ ਵਧਾਉਣਾ ਕਿੰਨਾ ਆਸਾਨ ਹੈ। ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ **BudgetReviewer** ਏਜੰਟ ਜੋੜਦੇ ਹਾਂ ਜੋ ਯਾਤਰੀ ਦੇ ਬਜਟ ਦੇ ਮੁਕਾਬਲੇ ਯੋਜਨਾ ਦੀ ਜਾਂਚ ਕਰਦਾ ਹੈ, ਉਹ ਚੀਜ਼ਾਂ ਜਿਨ੍ਹਾਂ ਨਾਲ ਖਰਚ ਆਲੇ-ਦੁਆਲੇ ਹੋ ਸਕਣ ਦਾ ਸੰਕੇਤ ਦਿੰਦਾ ਹੈ, ਅਤੇ ਪੈਸੇ ਬਚਾਉਣ ਵਾਲੇ ਵਿਕਲਪ ਸੁਝਾਉਂਦਾ ਹੈ। ਵਰਕਫਲੋ ਹੁਣ ਤਿੰਨ ਏਜੰਟ ਕ੍ਰਮਵਾਰ ਚਲਾਉਂਦਾ ਹੈ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਸਾਰांश

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿਖਿਆ ਕਿ ਕਿਵੇਂ:

1. **ਮਾਹਿਰ ਏਜੰਟ ਬਣਾਓ** — ਹਰ ਇੱਕ ਦਾ ਇੱਕ ਕੇਂਦ੍ਰਿਤ ਭੂਮਿਕਾ (ਯੋਜਨਾ ਬਣਾਉਣ, ਕੋਨਸੀਅਰਜ, ਬਜਟ ਸਮੀਖਿਆ) ਹੁੰਦੀ ਹੈ।
2. **ਏਜੰਟਾਂ ਨੂੰ ਕ੍ਰਮਿਕ ਕਾਰਜ ਪ੍ਰਵਾਹ ਵਿੱਚ ਜੋੜੋ** `WorkflowBuilder` ਅਤੇ `add_edge` ਦੀ ਵਰਤੋਂ ਕਰਕੇ।
3. **ਬਹੁ-ਏਜੰਟ ਪਾਈਪਲਾਈਨ ਤੋਂ ਆਉਟਪੁੱਟ ਸਟ੍ਰੀਮ ਕਰੋ**, ਟਰੈਕ ਕਰਨਾ ਕਿ ਕਿਹੜਾ ਏਜੰਟ ਬੋਲ ਰਿਹਾ ਹੈ।
4. **ਕਾਰਜ ਪ੍ਰਵਾਹ ਨੂੰ ਵਧਾਓ** ਨਵੇਂ ਏਜੰਟਾਂ ਨੂੰ ਕੜੀ ਵਿੱਚ ਸ਼ਾਮਲ ਕਰਕੇ ਬਿਨਾਂ ਮੌਜੂਦਾ ਏਜੰਟਾਂ ਨੂੰ ਬਦਲੇ।

ਬਹੁ-ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਹਰ ਏਜੰਟ ਨੂੰ ਸਧਾਰਣ ਰੱਖਦਾ ਹੈ ਜਦੋਂ ਕਿ ਇੱਕੱਲੇ ਕਿਸੇ ਵੀ ਏਜੰਟ ਦੀ ਤੁਲਨਾ ਵਿੱਚ ਜ਼ਿਆਦਾ ਧਨਾਢ੍ਹ ਅਤੇ ਵੀਚਾਰ-ਵਿਮਰਸ਼ਯੋਗ ਨਤੀਜੇ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
